In [6]:
import torch 
import torch.nn as nn
import numpy as np
import torch.optim as optim
import pandas as pd


In [7]:
agents = pd.DataFrame(np.random.uniform(0, 1, size = (500, 2)), columns=['x', 'y'])
agents['orange']=1
agents.loc[250:, 'orange']=0

agents.head()

,x,y,orange
0,0.264689,0.345430,1
1,0.446564,0.300772,1
2,0.442565,0.815363,1
3,0.469290,0.516118,1
4,0.106462,0.683121,1


In [8]:
unhappy=1
iteration = 0
while unhappy>0:
    moved=[]
    agents_temp=agents.copy()
    for agent in agents.index.tolist():
    
        diff = agents_temp[['x', 'y']]-agents_temp.loc[agent, ['x', 'y']]
        sq_diff = diff**2
        sum_sq_diff = sq_diff.sum(axis=1)
        
        sqrt_sum_sq_diff = sum_sq_diff**0.5
        agents_temp['sqrt_sum_sq_diff']=sqrt_sum_sq_diff
        
        if sum(agents_temp.sort_values(by='sqrt_sum_sq_diff')['orange'][1:11]==agents_temp.loc[agent, 'orange'])<5:
            moved.append(agent)
            agents.loc[agent, ['x', 'y']]=np.random.uniform(0, 1, size = (1, 2))[0]        
    unhappy=len(moved)
    iteration += 1  
    print('On iteration: '+str(iteration)+': '+str(len(moved))+' agets moved')

On iteration: 1: 188 agets moved
On iteration: 2: 119 agets moved
On iteration: 3: 67 agets moved
On iteration: 4: 53 agets moved
On iteration: 5: 30 agets moved
On iteration: 6: 19 agets moved
On iteration: 7: 10 agets moved
On iteration: 8: 3 agets moved
On iteration: 9: 4 agets moved
On iteration: 10: 3 agets moved
On iteration: 11: 1 agets moved
On iteration: 12: 2 agets moved
On iteration: 13: 1 agets moved
On iteration: 14: 1 agets moved
On iteration: 15: 0 agets moved


In [5]:
# ----------------- Config -----------------
N = 500
K = 10
MOVE_PENALTY = 0.02      # small cost to move
UNHAPPY_PENALTY = 0.8    # extra cost if still unhappy
TRAIN_STEPS = 8000
SEED = 7
np.random.seed(SEED); torch.manual_seed(SEED)

# ----------------- Environment -----------------
class Env:
    def __init__(self, n=N, k=K, move_penalty=MOVE_PENALTY, unhappy_penalty=UNHAPPY_PENALTY):
        self.n, self.k = n, k
        self.move_penalty, self.unhappy_penalty = move_penalty, unhappy_penalty
        self.types = np.random.randint(0, 2, n)
        self.pos = np.random.rand(n, 2)

    def obs(self):
        D = np.sqrt(((self.pos[:,None,:]-self.pos[None,:,:])**2).sum(2))
        idx = np.argsort(D)[:, 1:self.k+1]
        same = (self.types[idx] == self.types[:,None]).mean(1).astype(np.float32)
        dens = D[np.arange(self.n)[:,None], idx].mean(1).astype(np.float32)
        dens = dens / (dens.mean() + 1e-8)  # light normalization
        return np.stack([same, dens], 1)    # shape [N,2]

    def step(self, act):
        move = (act == 1)
        if move.any():
            self.pos[move] = np.random.rand(move.sum(), 2)
        o = self.obs()
        satisfied = (o[:,0] >= 0.5)
        r = satisfied.astype(np.float32) - self.move_penalty * move.astype(np.float32)
        r[~satisfied] -= self.unhappy_penalty
        return o, r, int(move.sum()), satisfied

# ----------------- Policy -----------------
class Policy(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 16), nn.Tanh(),
            nn.Linear(16, 2)             # logits: [stay, move]
        )
    def forward(self, x): return self.net(x)

# ----------------- Train (REINFORCE) -----------------
env = Env()
pol = Policy()
opt = optim.Adam(pol.parameters(), lr=1e-3)

print("Initial satisfied_rate:", (env.obs()[:,0] >= 0.5).mean())

for step in range(TRAIN_STEPS):
    o = torch.from_numpy(env.obs())
    logits = pol(o)
    dist = torch.distributions.Categorical(logits=logits)
    a = dist.sample()
    logp = dist.log_prob(a)

    _, r_np, _, _ = env.step(a.numpy())
    r = torch.from_numpy(r_np)

    adv = (r - r.mean()) / (r.std() + 1e-6)
    loss = -(logp * adv).mean()
    opt.zero_grad(); loss.backward(); opt.step()

    if (step+1) % 1000 == 0:
        print(f"train step {step+1}/{TRAIN_STEPS} | mean_r={r.mean().item():.3f}")

# ----------------- Greedy run until stable -----------------
for i in range(300):
    o = torch.from_numpy(env.obs())
    a = torch.argmax(pol(o), 1).numpy()       # greedy: stay vs move
    _, _, moved, satisfied = env.step(a)
    sat_rate = satisfied.mean()
    if moved == 0 and satisfied.all():
        print(f"Stable after {i} iterations | satisfied_rate={sat_rate:.3f}")
        break
    if i % 10 == 0:
        print(f"iter {i}: moved={moved}, satisfied_rate={sat_rate:.3f}")
else:
    print("Did not fully stabilize within the iteration limit.")


Initial satisfied_rate: 0.61
train step 1000/8000 | mean_r=1.000
train step 2000/8000 | mean_r=0.996
train step 3000/8000 | mean_r=1.000
train step 4000/8000 | mean_r=1.000
train step 5000/8000 | mean_r=1.000
train step 6000/8000 | mean_r=1.000
train step 7000/8000 | mean_r=1.000
train step 8000/8000 | mean_r=0.996
iter 0: moved=0, satisfied_rate=0.998
iter 10: moved=0, satisfied_rate=0.998
iter 20: moved=0, satisfied_rate=0.998
iter 30: moved=0, satisfied_rate=0.998
iter 40: moved=0, satisfied_rate=0.998
iter 50: moved=0, satisfied_rate=0.998
iter 60: moved=0, satisfied_rate=0.998
iter 70: moved=0, satisfied_rate=0.998
iter 80: moved=0, satisfied_rate=0.998
iter 90: moved=0, satisfied_rate=0.998
iter 100: moved=0, satisfied_rate=0.998
iter 110: moved=0, satisfied_rate=0.998
iter 120: moved=0, satisfied_rate=0.998
iter 130: moved=0, satisfied_rate=0.998
iter 140: moved=0, satisfied_rate=0.998
iter 150: moved=0, satisfied_rate=0.998
iter 160: moved=0, satisfied_rate=0.998
iter 170: move